# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**

[FAIR² Croissant JSON-LD Schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.date_published if hasattr(metadata, 'date_published') else getattr(metadata, 'datePublished', 'n/a')}")
print(f"Citation: {metadata.cite_as}")


## 2. Data Overview
Review available record sets and fields. All entities are referenced by their `@id` fields for clarity and reproducibility.

In [ ]:
# List all record sets in the dataset with their @id and available fields
record_sets = list(metadata.record_sets)
if not record_sets:
    print("No record sets found in metadata (some Croissant datasets may define them dynamically).")
else:
    print("Available record sets and fields:")
    for rs in record_sets:
        print(f"- Record Set ID: {rs.id}")
        print(f"  Name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
        field_ids = [f.id for f in getattr(rs, 'fields', [])]
        print(f"  Fields: {field_ids}")

# For this example, we attempt data access dynamically (since many Croissant datasets have a single major record set)
# We'll use mlcroissant's JSON-LD parser to find the likely table-oriented record set if not explicitly listed.
if not record_sets:
    # Some datasets expose record_sets with .record_set instead
    rs_ids = getattr(metadata, 'record_set', [])
    if rs_ids:
        print(f"Found record set identifiers: {rs_ids}")


## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use the `@id` values captured above. Here, we load all record sets if available.

In [ ]:
# Gather all record set IDs (by @id)
record_set_ids = [rs.id for rs in getattr(metadata, 'record_sets', [])]  # use .record_sets if available

# If record_set is empty, try .record_set (list of @id strings)
if not record_set_ids:
    record_set_ids = getattr(metadata, 'record_set', [])
# If still empty, use default ('main', most Croissant schemas have one main record set)
if not record_set_ids:
    record_set_ids = ['main']

print(f"Record set IDs to extract: {record_set_ids}")
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from record set '{record_set_id}'")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Error accessing record set {record_set_id}: {e}")

# List columns of the first record set loaded
chosen_record_set_id = record_set_ids[0]
print(f"Columns for record set '{chosen_record_set_id}': {dataframes[chosen_record_set_id].columns.tolist()}")
dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. We select a numeric field and a group/categorical field (by their `@id`/column name) for demonstration.

In [ ]:
# List available columns to pick numeric/group fields
df = dataframes[chosen_record_set_id]
print("Available columns:", df.columns.tolist())

# --- Specify field IDs (columns) for analysis ---
# For demo, pick the first numeric (float/int) field and a suitable group field
numeric_candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
group_candidates = [c for c in df.columns if c.lower() in ['sample', 'group', 'type', 'category', 'label'] or 'group' in c.lower()]
print(f"Detected numeric fields: {numeric_candidates}")
print(f"Detected group fields: {group_candidates}")

# Select a numeric field (fallback to first column if type inference fails)
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.columns[0]

# Select a group field, if available
group_field = group_candidates[0] if group_candidates else None

print(f"Using numeric field: '{numeric_field}' (as @id reference)")
if group_field:
    print(f"Using group field: '{group_field}' (as @id reference)")

# --- Filtering, Normalizing, Grouping ---
try:
    # Filter operation (threshold at the 10th percentile for illustration, or a static value)
    threshold = df[numeric_field].quantile(0.1) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())

    # Grouping, if group field exists
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
except Exception as e:
    print(f"Could not perform EDA operations: {e}")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the selected numeric field, and if available, a group-wise mean.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# Plot normalized values if computed
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True)
    plt.title(f'Normalized Distribution of {numeric_field}')
    plt.xlabel(f"{numeric_field}_normalized")
    plt.show()

# Boxplot by group field if available
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, and analysis of the FAIR² dataset using the `mlcroissant` library, referencing all key dataset entities by their `@id`. You can now extend this workflow for in-depth domain-specific analysis, applying more advanced statistical techniques as needed.